In [ ]:
import requests
import pandas as pd
import pyodbc
import os
from datetime import datetime
from decimal import Decimal  


# 1. CONFIGURATION

HF_TOKEN = os.getenv('HF_TOKEN', 'YOUR_HF_TOKEN')
SERVER = r'jk\SQLEXPRESS'
DATABASE = 'CryptoProject'
DRIVER = '{ODBC Driver 17 for SQL Server}'
API_URL = "https://api-inference.huggingface.co/models/mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis"


# 2. EXTRACTION

def extract_crypto_data():
    print("Fetching live market data...")
    url = "https://api.coingecko.com/api/v3/coins/markets"
    params = {'vs_currency': 'usd', 'order': 'market_cap_desc', 'per_page': 50, 'sparkline': 'false'}
    response = requests.get(url, params=params)
    return pd.DataFrame(response.json())


# 3. TRANSFORMATION

def get_sentiment(text, headers):
    try:
        res = requests.post(API_URL, headers=headers, json={"inputs": text}).json()
        return res[0][0]['label']
    except:
        return "Neutral"

def transform_data(df):
    print("Calculating benchmarks and AI sentiment...")
    df['benchmark_value'] = df.head(5)['market_cap'].mean()

    headers = {"Authorization": f"Bearer {HF_TOKEN}"}
    df['sentiment_score'] = df['name'].apply(lambda x: get_sentiment(f"Market outlook for {x} is shifting.", headers))
    return df


# 4. LOAD (NATIVE DECIMAL CASTING)

def safe_decimal(val, decimals):
    if pd.isna(val) or val is None:
        return None
    fmt = f"{{:.{decimals}f}}"
    return Decimal(fmt.format(float(val)))

def safe_int(val):
    if pd.isna(val) or val is None:
        return None
    return int(val)

def load_to_sql(df):
    print("Loading data into SQL Server...")
    conn_str = f'DRIVER={DRIVER};SERVER={SERVER};DATABASE={DATABASE};Trusted_Connection=yes;'
    conn = pyodbc.connect(conn_str)
    cursor = conn.cursor()

    for _, row in df.iterrows():
        cursor.execute("""
            INSERT INTO crypto_prices (
                id, symbol, name, current_price, market_cap, market_cap_rank, 
                total_volume, high_24h, low_24h, price_change_percentage_24h, 
                sentiment_score, benchmark_value
            ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?)""",
            row.id, 
            row.symbol, 
            row.name, 
            safe_decimal(row.current_price, 8), 
            safe_int(row.market_cap), 
            safe_int(row.market_cap_rank),
            safe_int(row.total_volume), 
            safe_decimal(row.high_24h, 8), 
            safe_decimal(row.low_24h, 8), 
            safe_decimal(row.price_change_percentage_24h, 5),
            row.sentiment_score, 
            safe_decimal(row.benchmark_value, 2)
        )
    
    conn.commit()
    conn.close()
    print(f"ETL Complete! Snapshot saved at {datetime.now()}")


# 5. EXECUTION

if __name__ == "__main__":
    raw_data = extract_crypto_data()
    clean_data = transform_data(raw_data)
    load_to_sql(clean_data)

Fetching live market data...
Calculating benchmarks and AI sentiment...
Loading data into SQL Server...
ETL Complete! Snapshot saved at 2026-06-02 19:27:05.238813


CONTINUOUS 15-SECOND BACKGROUND LOOP

In [ ]:
import time
import sys

if __name__ == "__main__":
    INTERVAL = 15  # Strict 15-second loop window
    error_counter = 0
    
    print(f"[SYSTEM] Engine active. Monitoring 50 digital assets every {INTERVAL}s.")
    print("[SYSTEM] Running windowless background worker. Stream active...\n")
    
    while True:
        try:
            # Execute full pipeline steps sequentially
            raw_data = extract_crypto_data()
            clean_data = transform_data(raw_data)
            load_to_sql(clean_data)
            
            # Reset error counter upon a completely successful write cycle
            error_counter = 0 
            time.sleep(INTERVAL)
            
        except Exception as e:
            error_counter += 1
            # If the API hits a 429 rate limit or network drops, back off gracefully
            backoff_time = min(15 * error_counter, 300) 
            print(f"[ERR] Cycle failure at {datetime.now()}. Error details: {e}")
            print(f"[ERR] Cooling down pipeline. Automatically retrying in {backoff_time}s...")
            time.sleep(backoff_time)
            continue